In [8]:
import pandas as pd
import anthropic

In [10]:
client = anthropic.Anthropic()

# Load input
df = pd.read_csv('/app/data/batch_sentiment_input.csv')
print(f"Reviews to process: {len(df):,}")
print(f"Unique review_ids: {df['review_id'].nunique():,}")

Reviews to process: 41,940
Unique review_ids: 41,940


In [11]:
df = df.reset_index(drop=True)
df['batch_index'] = df.index

# Save mapping table locally
df[['batch_index', 'review_id']].to_csv('/app/data/sentiment_batch_mapping.csv', index=False)
print(f"Mapping saved to /app/data/sentiment_batch_mapping.csv")

Mapping saved to /app/data/sentiment_batch_mapping.csv


In [12]:
requests = []
for _, row in df.iterrows():
    requests.append({
        "custom_id": f"sentiment_{row['batch_index']}",
        "params": {
            "model": "claude-haiku-4-5-20251001",
            "max_tokens": 10,
            "messages": [{
                "role": "user",
                "content": f"""Analyze the sentiment of this e-commerce review.
Respond with exactly one word only: positive, negative, or neutral. No explanation.

Review: {row['combined_text']}"""
            }]
        }
    })

print(f"Total requests: {len(requests):,}")

# Submit batch
batch = client.messages.batches.create(requests=requests)
#print(f"Batch ID: {batch.id}")
print(f"Status: {batch.processing_status}")

# Save batch ID
with open('/app/data/sentiment_batch_id.txt', 'w') as f:
    f.write(batch.id)
print("Batch ID saved to /app/data/sentiment_batch_id.txt")

Total requests: 41,940
Batch ID: msgbatch_0163bhxdySU3HDsJbmVpGyFm
Status: in_progress
Batch ID saved to /app/data/sentiment_batch_id.txt
